In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_city_district_data(city_name):
    """
    获取指定城市的区域和总价数据。
    """
    conn = get_db_connection()
    # 使用 f-string 绕过部分 PyMySQL 版本对 StarRocks 传参的兼容性问题
    query = f"""
    SELECT 
        CASE 
            WHEN district IS NOT NULL AND district != '' THEN district
            WHEN neighborhood IS NOT NULL AND neighborhood != '' THEN neighborhood
            ELSE '未知区域'
        END AS district,
        total_price 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND total_price IS NOT NULL 
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_stacked_area(city):
    """绘制百分比堆叠面积图"""
    df = fetch_city_district_data(city)
    
    if df.empty or len(df) < 50:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, f"{city} 数据量不足 (当前行数: {len(df)})，无法生成有效的结构分布图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 自适应价格区间划分 (动态分箱) ---
    quantiles = df['total_price'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
    bins = [-1] + [q + (i * 0.0001) for i, q in enumerate(quantiles)] + [float('inf')]
    
    labels = [
        f'低价位 (<{int(quantiles[0])}万)', 
        f'中低价 ({int(quantiles[0])}-{int(quantiles[1])}万)', 
        f'中等价 ({int(quantiles[1])}-{int(quantiles[2])}万)', 
        f'中高价 ({int(quantiles[2])}-{int(quantiles[3])}万)', 
        f'高价位 (>{int(quantiles[3])}万)'
    ]
    df['price_tier'] = pd.cut(df['total_price'], bins=bins, labels=labels)

    # --- 2. 按区域聚合与排序 ---
    district_median = df.groupby('district')['total_price'].median().sort_values()
    sorted_districts = district_median.index.tolist()

    # 计算占比
    cross_tab = pd.crosstab(df['district'], df['price_tier'], normalize='index') * 100
    cross_tab = cross_tab.reindex(sorted_districts)

    # 降低剔除阈值，保留至少有 10 套房源的区域
    district_counts = df['district'].value_counts()
    valid_districts = [d for d in sorted_districts if district_counts.get(d, 0) >= 10]
    cross_tab = cross_tab.loc[valid_districts]

    if cross_tab.empty or len(cross_tab) < 2:
         plt.figure(figsize=(12, 6))
         plt.text(0.5, 0.5, f"{city} 有效分类维度不足 (如全集中在'未知区域')，无法绘制面积图", ha='center', va='center', fontsize=14)
         plt.axis('off')
         plt.show()
         return

    # --- 3. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = cross_tab.index
    y = [cross_tab[col] for col in labels]
    
    colors = sns.color_palette("RdYlBu_r", len(labels))

    ax.stackplot(x, y, labels=labels, colors=colors, alpha=0.85, edgecolor='white', linewidth=0.5)

    # --- 4. 图表修饰 ---
    plt.title(f'[{city}] 区域房价结构推移规律 (按区域均价由低到高排序 )', fontsize=16, pad=20)
    plt.xlabel('行政区 / 板块', fontsize=12, labelpad=10)
    plt.ylabel('各价格区间房源占比 (%)', fontsize=12)
    
    ax.set_ylim(0, 100)
    ax.margins(x=0) 
    
    plt.xticks(rotation=45, ha='right')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title="自适应价格区间")
    
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    sns.despine(left=True, bottom=True)

    plt.tight_layout()
    plt.show()

In [14]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_stacked_area(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_stacked_area(default_city)

In [15]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()